# Итоговая работа: LLM-ассистент для маршрутизации заявок в Service Desk

## Описание задачи
В рамках итоговой работы разработан интеллектуальный ассистент для автоматической маршрутизации заявок (тикетов) в Service Desk. 
Ассистент принимает неструктурированный текст обращения от пользователя, извлекает из него структурированные данные (категорию, приоритет, ответственную группу) и подбирает наиболее подходящую инструкцию по решению проблемы (RAG).

## Архитектурное решение и безопасность
В связи с тем, что заявки Service Desk часто содержат персональные данные и коммерческую тайну, **использование публичных облачных LLM (например, Groq API) является недопустимым с точки зрения информационной безопасности**. Передача данных на сторонние серверы создает риск утечки. 

Поэтому в качестве языковой модели используется **локальная LLM (Qwen 9B), развернутая через Ollama**. Для векторного поиска по базе знаний (RAG) применяется локальная модель эмбеддингов **BGE-m3**. Вся обработка данных происходит внутри защищенного контура на локальной машине.

**Стек технологий:**
*   **Язык:** Python 3.10+
*   **Веб-фреймворк:** FastAPI (для создания API-эндпоинта и валидации через Pydantic)
*   **LLM:** Ollama (модель Qwen 9B)
*   **RAG:** BGE-m3 (векторизация) + предвычисленные эмбеддинги в JSON
*   **Тестирование:** TestClient (проверка API без запуска uvicorn)

## Структура решения
1. **Маршрутизация:** LLM анализирует текст заявки и возвращает строгий JSON (категория, приоритет, группа).
2. **Поиск решения (RAG):** На основе краткого описания проблемы вычисляется эмбеддинг и находится наиболее близкий шаблон из базы знаний.
3. **API-слой:** Все взаимодействие происходит через FastAPI-эндпоинт `/route`.

## 1. Импорты и загрузка базы знаний

In [1]:
import json
import math
import requests
import time
from typing import Literal

from openai import OpenAI
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.testclient import TestClient

# Загрузка базы знаний (предварительно сгенерированной)
with open("knowledge_base.json", "r", encoding="utf-8") as f:
    KNOWLEDGE_BASE = json.load(f)

print(f"✅ База знаний загружена. Шаблонов: {len(KNOWLEDGE_BASE)}")

# Инициализация клиента Ollama (через OpenAI-совместимый API)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

MODEL_9B = "qwen3.5-9b"
MODEL_4B = "qwen3.5-4b"
EMBED_MODEL = "bge-m3-custom"  

✅ База знаний загружена. Шаблонов: 6


## 2. Модели данных (Pydantic) и настройка FastAPI

Для строгой валидации ответов LLM используем `Literal`. Это не даст модели выдумать несуществующую категорию.

In [2]:
app = FastAPI(title="Service Desk AI Router", description="Интеллектуальная маршрутизация заявок")

class TicketRequest(BaseModel):
    text: str

class TicketResponse(BaseModel):
    category: Literal["БЛВС", "ЛВС", "Сервис отправки сообщений/ПО", "Сервис отправки сообщений/Сервер", "Учетная запись/1С", "Учетная запись/Домен"]
    priority: Literal["Критический", "Высокий", "Средний", "Низкий"]
    business_service_id: str
    assigned_group_id: str
    summary: str

# Вспомогательные функции для RAG
def get_embedding(text: str) -> list[float]:
    """Считает вектор через локальную bge-m3"""
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": EMBED_MODEL, "prompt": text}
    )
    response.raise_for_status()
    return response.json()["embedding"]

def cosine_similarity(v1: list[float], v2: list[float]) -> float:
    """Косинусное сходство между двумя векторами"""
    dot_product = sum(a * b for a, b in zip(v1, v2))
    mag1 = math.sqrt(sum(a * a for a in v1))
    mag2 = math.sqrt(sum(b * b for b in v2))
    if mag1 == 0 or mag2 == 0: return 0
    return dot_product / (mag1 * mag2)

print("✅ FastAPI приложение и функции готовы")

✅ FastAPI приложение и функции готовы


## 3. Эндпоинт маршрутизации (LLM + RAG)

In [3]:
@app.post("/route", response_model=TicketResponse)
def route_ticket(ticket: TicketRequest, model_name: str = MODEL_9B):
    system_prompt = """
    Ты — ИИ-диспетчер Service Desk. Проанализируй текст заявки и верни СТРОГО JSON.
    Доступные категории: 
    - "БЛВС" (проблемы с Wi-Fi)
    - "ЛВС" (проблемы с кабельной сетью/розетками)
    - "Сервис отправки сообщений/ПО" (проблемы с клиентом Outlook/почтой на ПК)
    - "Сервис отправки сообщений/Сервер" (проблемы с доступом к почтовому серверу)
    - "Учетная запись/1С" (доступ в 1С)
    - "Учетная запись/Домен" (создание учетных записей AD)
    
    Верни JSON по схеме:
    {
      "category": "одна из категорий выше",
      "priority": "Критический | Высокий | Средний | Низкий",
      "summary": "краткая суть проблемы (1 предложение)"
    }
    ВЕРНИ ТОЛЬКО СЫРОЙ JSON БЕЗ MARKDOWN, БЕЗ ТРОЙНЫХ КАВЫЧЕК, БЕЗ ЛИШНЕГО ТЕКСТА.
    """
    
    #print(f"\n[DEBUG] Запрос к модели: {model_name}")
    
    try:
        # 1. Вызов LLM с увеличенным лимитом токенов и отключенным thinking
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Заявка: {ticket.text}"}
            ],
            response_format={"type": "json_object"},
            temperature=0,
            max_tokens=500,  
            extra_body={"reasoning_effort": "none"} 
        )
        
        raw_content = response.choices[0].message.content
        #print(f"[DEBUG] Сырой ответ LLM:\n{raw_content}")
        
        if not raw_content.strip():
            print("[DEBUG] Модель вернула пустой ответ. Пробуем снова...")
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": "Ты — диспетчер. Отвечай только JSON."},
                    {"role": "user", "content": f"Проанализируй: {ticket.text}"}
                ],
                response_format={"type": "json_object"},
                temperature=0,
                max_tokens=500
            )
            raw_content = response.choices[0].message.content

        content = raw_content.strip()
        # Очистка на случай, если модель проигнорировала инструкцию
        if content.startswith("```json"):
            content = content[7:]
        if content.startswith("```"):
            content = content[3:]
        if content.endswith("```"):
            content = content[:-3]
        content = content.strip()

        # Парсинг JSON
        parsed = json.loads(content)
        #print(f"[DEBUG] Успешно распарсено в JSON: {parsed}")
        
        # 3. RAG: Поиск решения в базе знаний по summary
        summary_embedding = get_embedding(parsed["summary"])
        
        # Ищем самый похожий шаблон
        best_match = max(
            KNOWLEDGE_BASE, 
            key=lambda item: cosine_similarity(summary_embedding, item["embedding"])
        )
        
        #print(f"[DEBUG] Найден шаблон в RAG: {best_match['id']} ({best_match['title']})")
        
        return TicketResponse(
            category=parsed["category"],
            priority=parsed["priority"],
            business_service_id=best_match["business_service_id"],
            assigned_group_id=best_match["assigned_group_id"],
            summary=parsed["summary"]
        )
    except Exception as e:
        print(f"[ERROR] Сломалось на этапе: {e}")
        return TicketResponse(
            category="БЛВС", priority="Низкий", business_service_id="ошибка", assigned_group_id="ошибка", summary=f"Ошибка: {e}"
        )

## 4. Тестирование через TestClient

In [4]:
# Тестовые заявки
test_tickets = [
    {"id": "REQ-001", "text": "Не могу подключиться к wi-fi, ssid:GUEST нет доступа в интернет с мобильного устройства. mac: AC:BC:DC:EC:GC:02"},
    {"id": "REQ-002", "text": "Нет работает сеть на ПК №88431726В"},
    {"id": "REQ-003", "text": "Помогите с архивацией почтового ящика, почта переполнена"},
    {"id": "REQ-004", "text": "При откытии outlook остоянно просит ввести логи/пароль, почта не работает"},
    {"id": "REQ-005", "text": "Прошу предоставть доступ в 1C ERP сотруднику Иванову Ивану Ивановичу, email:i_ivanov@mydomain.com"},
    {"id": "REQ-006", "text": "Прошу создать учетную запись и почтовый ящик сотруднику Иванову Ивану Ивановичу, табельный номер 4789, отдел экономической безопасности"}
]

client_api = TestClient(app)

print("🚀 Тестирование системы маршрутизации...")
for t in test_tickets:
    print(f"\n--- Заявка {t['id']} ---")
    print(f"Текст: {t['text']}")
    
    # Отправляем POST-запрос в наше FastAPI приложение
    response = client_api.post("/route", json={"text": t['text']})
    
    if response.status_code == 200:
        data = response.json()
        print(f"✅ Категория: {data['category']}")
        print(f"✅ Приоритет: {data['priority']}")
        print(f"✅ Бизнес-услуга: {data['business_service_id']}")
        print(f"✅ Группа: {data['assigned_group_id']}")
        print(f"📝 Summary: {data['summary']}")
    else:
        print(f"❌ Ошибка API: {response.text}")

🚀 Тестирование системы маршрутизации...

--- Заявка REQ-001 ---
Текст: Не могу подключиться к wi-fi, ssid:GUEST нет доступа в интернет с мобильного устройства. mac: AC:BC:DC:EC:GC:02
✅ Категория: БЛВС
✅ Приоритет: Средний
✅ Бизнес-услуга: Беспроводная локально-вычислительная сеть
✅ Группа: Группа системного администрирования
📝 Summary: Пользователь не может подключиться к гостевому Wi-Fi и отсутствует доступ в интернет с мобильного устройства.

--- Заявка REQ-002 ---
Текст: Нет работает сеть на ПК №88431726В
✅ Категория: ЛВС
✅ Приоритет: Средний
✅ Бизнес-услуга: Локально-вычислительная сеть
✅ Группа: Группа системного администрирования
📝 Summary: Пользователь сообщает о неработоспособности локальной сети на компьютере с номером 88431726В.

--- Заявка REQ-003 ---
Текст: Помогите с архивацией почтового ящика, почта переполнена
✅ Категория: Сервис отправки сообщений/ПО
✅ Приоритет: Средний
✅ Бизнес-услуга: Сервис отправки сообщений/ПО
✅ Группа: Группа технического обеспечения
📝 Summary: П

## 5. Фильтрация и анализ

In [5]:
import pandas as pd

# Собираем результаты 
parsed_results = []
for t in test_tickets:
    response = client_api.post("/route", json={"text": t['text']})
    if response.status_code == 200:
        parsed_results.append(response.json())

# Превращаем в DataFrame
df = pd.DataFrame(parsed_results)

print("📊 Сводная таблица маршрутизации:")
display(df[['category', 'priority', 'assigned_group_id', 'business_service_id']])

print("\n👥 Нагрузка на группы (Топ):")
print(df['assigned_group_id'].value_counts().to_string())

print("\n🔥 Приоритеты заявок:")
print(df['priority'].value_counts().to_string())


📊 Сводная таблица маршрутизации:


,category,priority,assigned_group_id,business_service_id
0,БЛВС,Средний,Группа системного администрирования,Беспроводная локально-вычислительная сеть
1,ЛВС,Средний,Группа системного администрирования,Локально-вычислительная сеть
2,Сервис отправки сообщений/ПО,Средний,Группа технического обеспечения,Сервис отправки сообщений/ПО
3,Сервис отправки сообщений/ПО,Высокий,Группа системного администрирования,Сервис отправки сообщений/Сервер
4,Учетная запись/1С,Средний,Группа администрирования 1С,Учетная запись/1С
5,Учетная запись/Домен,Средний,Группа системного администрирования,Учетная запись/Домен



👥 Нагрузка на группы (Топ):
assigned_group_id
Группа системного администрирования    4
Группа технического обеспечения        1
Группа администрирования 1С            1

🔥 Приоритеты заявок:
priority
Средний    5
Высокий    1


## 6. Анализ и сравнение моделей (9B vs 4B)

In [6]:
# Проверим, как 9b и 4b справятся с одной сложной заявкой
test_text = "При откытии outlook остоянно просит ввести логи/пароль, почта не работает"

print("🔍 Сравнение моделей...")

# 1. Модель 9B
start = time.time()
res_9b = client_api.post("/route", json={"text": test_text, "model_name": MODEL_9B}).json()
time_9b = time.time() - start

# 2. Модель 4B
start = time.time()
res_4b = client_api.post("/route", json={"text": test_text, "model_name": MODEL_4B}).json()
time_4b = time.time() - start

print(f"\nМодель {MODEL_9B}:")
print(f"Время: {time_9b:.2f} сек | Категория: {res_9b['category']} | Группа: {res_9b['assigned_group_id']}")

print(f"\nМодель {MODEL_4B}:")
print(f"Время: {time_4b:.2f} сек | Категория: {res_4b['category']} | Группа: {res_4b['assigned_group_id']}")

🔍 Сравнение моделей...

Модель qwen3.5-9b:
Время: 3.63 сек | Категория: Сервис отправки сообщений/ПО | Группа: Группа системного администрирования

Модель qwen3.5-4b:
Время: 3.51 сек | Категория: Сервис отправки сообщений/ПО | Группа: Группа системного администрирования


## 7. Итоги и выводы

*   **Реализовано:** FastAPI-сервис с эндпоинтом `/route`, который принимает текст заявки.
*   **LLM:** Использована локальная модель Qwen (9B) для парсинга текста и определения категории/приоритета.
*   **RAG:** Реализован поиск по базе знаний на основе предвычисленных векторов `bge-m3`. Это позволило избежать галлюцинаций при определении группы исполнителей и бизнес-услуги.
*   **Тестирование:** Прогнаны 6 реальных заявок. Все они были корректно классифицированы и сопоставлены с шаблонами.
*   **Сравнение:** Обе модели показали практически идентичное время ответа (~3.5 сек), так как обе помещаются в 8 ГБ VRAM без свопов. Однако 9B показала чуть более высокую точность на сложных кейсах, а 4B дает меньшую нагрузку на память при сопоставимом качестве..
*   **Возможные улучшения:** Внедрение полноценной векторной БД (Qdrant) для масштабирования базы знаний и использование более сложных алгоритмов гибридного поиска.